# Setup

In [ ]:
!pip install xarray tiler rioxarray  # For H100 system

In [ ]:
# Standard library imports
import os
import subprocess
import sys
from datetime import datetime
from glob import glob
from pathlib import Path

# Third-party imports
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import numpy.ma as ma
import torch
import xarray as xr
from tqdm import tqdm
from transformers import AutoImageProcessor
from tiler import Tiler, Merger

%matplotlib inline

In [ ]:
repo_dir = "lfm"

# if not os.path.exists(repo_dir):
#     subprocess.run(["git", "clone", f"https://github.com/nasa-nccs-hpda/{repo_dir}"])
# else:
#     subprocess.run(["git", "-C", repo_dir, "pull"])

In [ ]:
# sys.path.insert(0, os.path.join(os.getcwd(), repo_dir))
sys.path.insert(0, "/explore/nobackup/people/ajkerr1/Lunar_FM/latest_repo/lfm")

from lfm.tasks.sem_segmentation.sseg_model import load_dinov3_encoder, DINOSegmentation
from lfm.tasks.sem_segmentation.data_cube_inference import run_datacube_inference, calculate_datacube_statistics

# Config

In [ ]:
# Data paths
INPUT_DIR = "/explore/nobackup/projects/lfm/model_inputs/data_cubes/WAC_STATIC"

# Where to load dinov3 init weights
weights_local_checkpoint = '/explore/nobackup/projects/ilab/models/dinov3/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth'
pretrained_checkpoint = "/explore/nobackup/projects/lfm/model_inference/checkpoints/sem_seg/12_band/checkpoint_epoch_100_NO_ZSCORE.pt"

# Output dir (this will be created automatically)
OUTPUT_DIR = Path("./outputs/cube_inference")  # Change this if you want a specific path
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
print(f"Output directory: {OUTPUT_DIR}")

# Training hyperparameters
BATCH_SIZE = 16
NUM_WORKERS = 8

# Number of bands, band filter
NUM_BANDS = 12  # n bands to be passed through model; NOT n bands in base input!
BAND_FILTER = None  # UV bands are first 2, RGB are bands 5, 3, 2

# Model parameters
TARGET_SIZE = (304, 304)  # Input size for DINO model
FREEZE_ENCODER = False

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load model

In [ ]:
# Load model from model factory
encoder = load_dinov3_encoder(weights_local_checkpoint, device=device)
model = DINOSegmentation(
    encoder=encoder,  # Use DINOv3 head
    num_classes=2,  # Binary segmentaton (crater, not crater)
    img_size=TARGET_SIZE,
    num_bands=NUM_BANDS,
).to(device)

# Apply checkpoint
checkpoint = torch.load(pretrained_checkpoint, map_location='cpu')
checkpoint_state = checkpoint['model_state_dict']
missing_keys, unexpected_keys = model.load_state_dict(
    checkpoint_state, strict=False
)
model.to(device)
model.eval()
print("Successfully loaded model from checkpoint!")

# Load data

## Load statistics from training to normalize inputs

In [ ]:
print("\n" + "="*60)
print("STEP 1: Loading training dataset statistics...")

# Load mean and std from a previous training run
MEAN = np.load("/explore/nobackup/projects/lfm/model_inference/stats/sem_seg/12_band/dataset_mean.npy")
STD = np.load("/explore/nobackup/projects/lfm/model_inference/stats/sem_seg/12_band/dataset_std.npy")

print("Done.")
print(f"Mean: {MEAN},\nSTD: {STD}")
print("="*60)

# Inference

In [ ]:
import numpy as np
n_images = 10
overlap = 0.75
thresh = 0.3
# output_dir = Path(f"{OUTPUT_DIR}/overlap_{int(overlap*100)}")
# output_dir.mkdir(exist_ok=True, parents=True)
output_path = f"{OUTPUT_DIR}/inference_viz_t_{int(thresh*100)}_d_06_12.png"
images, preds = run_datacube_inference(
    model=model,
    device=device,
    input_dir=INPUT_DIR,
    mean=MEAN,
    std=STD,
    output_path=output_path,
    n_images=n_images,
    model_native_size=TARGET_SIZE[0],
    tile_overlap=overlap,
    threshold=thresh,
    save_inputs_dir=None,
    debug=False,
    tile_window='hann',
    visualize_tiles=False
)